In [1]:
from data_preprocessing import data_preprocessing
from genetiec_algo import FeatureSelectionGeneticAlgorithm
from model_evaluation import evaluate
import datetime
import numpy as np
import pandas as pd
from statnew import Pearson, Distance
from classicalmethods import perform_rfe

## Preprocessing

In [ ]:
print("fetching & preprocessing...")
result = data_preprocessing(
    # csv_url="https://raw.githubusercontent.com/Amer-Zakaria/600-features-dataset/refs/heads/main/data.csv"
    )
print("done fetching & preprocessing the data!")


fetching & preprocessing...
used random sampling (no stratification)
Using knn imputation with  5  neighbors
 applied  robust  scaling
Final dataset:  1170  training,  391  test samples
features:  {474}
done fetching & preprocessing the data!


## Original Data Evaluation

In [3]:
# Obtain time & MSE on the original data
start_time = datetime.datetime.now()
original_mse = evaluate(
    result.X_train,
    result.X_test,
    result.y_train,
    result.y_test,
)
elapsed = datetime.datetime.now() - start_time
original_time = int(elapsed.total_seconds() * 1000)

print(f"Time & MSE of the original data: {original_time}ms & {original_mse}")


Time & MSE of the original data: 293ms & 229271323.25504762


## Run Genetic Algorithm

In [4]:
# Initialize the genetic algorithm
ga = FeatureSelectionGeneticAlgorithm(
    population_size=10,
    generations=18,
    mutation_rate=0.1,
    X_train=result.X_train,
    X_test=result.X_test,
    y_train=result.y_train,
    y_test=result.y_test,
)

# Run the genetic algorithm
(best_individual, best_fitness) = ga.solve()

# Obtain time & MSE after feature selection
start_time = datetime.datetime.now()
mask = np.array(best_individual, dtype=bool)
GA_mse = evaluate(
    result.X_train.loc[:, mask],
    result.X_test.loc[:, mask],
    result.y_train,
    result.y_test,
)
elapsed = datetime.datetime.now() - start_time
GA_time = int(elapsed.total_seconds() * 1000)

# Print results
print(f"Number of selected features: {sum(best_individual)} out of {result.X_train.shape[1]}")
print(f"Time & MSE after Genetic Alogrithm feature selection: {GA_time}ms & {GA_mse}")

Number of selected features: 213 out of 474
Time & MSE after Genetic Alogrithm feature selection: 54ms & 832.2750647018787


## Run Statisticals Methods

In [5]:
# Combine train and test data for statistical methods
X = pd.concat([result.X_train, result.X_test])
y = pd.concat([result.y_train, result.y_test])

# Calculate Pearson correlation and evaluate
PearsonResults = Pearson(X, y)

num_features_for_statistical_methods = 10
Pearson_selected_features = PearsonResults.index.tolist()[
    :num_features_for_statistical_methods
]

Pstart_eval = datetime.datetime.now()
Pearson_mse = evaluate(
    result.X_train.loc[:, Pearson_selected_features],
    result.X_test.loc[:, Pearson_selected_features],
    result.y_train,
    result.y_test,
)
Pearson_time_eval = int(
    (datetime.datetime.now() - Pstart_eval).total_seconds() * 1000
)

# Calculate Distance correlation and evaluate
DistanceResults = Distance(X, y)

Distance_selected_features = DistanceResults.index.tolist()[
    :num_features_for_statistical_methods
]

Dstart_eval = datetime.datetime.now()
Distance_mse = evaluate(
    result.X_train.loc[:, Distance_selected_features],
    result.X_test.loc[:, Distance_selected_features],
    result.y_train,
    result.y_test,
)
Distance_time_eval = int(
    (datetime.datetime.now() - Dstart_eval).total_seconds() * 1000
)

print("Pearson Correlation Feature Selection:")
print(f"  Number of selected features: {len(Pearson_selected_features)}")
print(f"  Evaluation Time: {Pearson_time_eval}ms")
print(f"  MSE: {Pearson_mse}.")
print("")
print( "Distance Correlation Feature Selection:")
print(f"  Number of selected features: {len(Distance_selected_features)}")
print(f"  Evaluation Time: {Distance_time_eval}ms")
print(f"  MSE: {Distance_mse}")

Pearson Correlation Feature Selection:
  Number of selected features: 10
  Evaluation Time: 57ms
  MSE: 5071.142228949703.

Distance Correlation Feature Selection:
  Number of selected features: 10
  Evaluation Time: 12ms
  MSE: 5236.962125173037


## Run Classical Method

In [6]:
no_of_features = 30

rfe_results = perform_rfe(
    split_data=result,
    n_features_to_select=no_of_features,  ## choose num of features to keep ##
    step=1  ## choose num features to remove at a time for speed (bigger ==> faster) ##
)

start_time = datetime.datetime.now()
rfe_mse = evaluate(
    result.X_train.loc[:, rfe_results.selected_features],
    result.X_test.loc[:, rfe_results.selected_features],
    result.y_train,
    result.y_test,
)
elapsed = datetime.datetime.now() - start_time
rfe_time = int(elapsed.total_seconds() * 1000)

print(f"Number of selected features: {no_of_features} out of {result.X_train.shape[1]}")
print(f"Time & MSE: {rfe_time}ms & {rfe_mse}")

Original features: 474 
Selected features: 30
Step size: 1 
Train RMSE: 519.6883
Test RMSE: 18157765.8976

SELECTED FEATURES:
136
70
50
62
45
43
60
154
48
68
12
427
18
16
453
409
36
34
177
448
181
447
176
174
172
309
307
249
387
114

TOP 10 FEATURE RANKINGS:
   1. 34 (SELECTED)
   1. 409 (SELECTED)
   1. 45 (SELECTED)
   1. 62 (SELECTED)
   1. 50 (SELECTED)
   1. 70 (SELECTED)
   1. 427 (SELECTED)
   1. 136 (SELECTED)
   1. 36 (SELECTED)
   1. 18 (SELECTED)
Number of selected features: 30 out of 474
Time & MSE: 6ms & 18157765.89755014
